<a href="https://colab.research.google.com/github/cemilkaracam/Haar-Wavelet-Regularized-Approximation-of-Discontinuous-Functions/blob/main/numeric_result.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Gerekirse: PyWavelets kurulumu (Google Colab'da kullanıyorsan aç)
# !pip install pywavelets

import numpy as np
import pywt

# Sürekli, basamak ve toplam fonksiyonlar
def f_continuous(x):
    return np.sin(4 * np.pi * x)

def f_discontinuous(x):
    return (x >= 0.5).astype(float)

def f_total(x):
    return f_continuous(x) + f_discontinuous(x)

# Haar ile sadece düşük seviye approximation (level=3'e kadar detayları sıfırla)
def haar_regularize(signal, level=3):
    # 'symmetric' mode ile sınır etkilerini kontrol ediyoruz
    coeffs = pywt.wavedec(signal, 'haar', level=level, mode='symmetric')
    # Tüm detay katsayılarını sıfırla (sadece approximation kalıyor)
    coeffs_trunc = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
    # Sinyali yeniden oluştur
    rec = pywt.waverec(coeffs_trunc, 'haar', mode='symmetric')
    return rec[:len(signal)]

# Izgara
x = np.linspace(0, 1, 1024, endpoint=False)  # endpoint=False çok önemli (dyadik yapı)
dx = x[1] - x[0]

# Gerçek hedef fonksiyon
f_true = f_total(x)

# 1) Unified (tek seferde hepsini regularize et)
f_unified = haar_regularize(f_true, level=3)
err_unified = np.sqrt(np.sum((f_true - f_unified)**2) * dx)

# 2) Decomposition (sadece basamak kısmını regularize et)
f_discont_smooth = haar_regularize(f_discontinuous(x), level=3)
f_decomp = f_continuous(x) + f_discont_smooth
err_decomp = np.sqrt(np.sum((f_true - f_decomp)**2) * dx)

# Sonuçları bilimsel gösterim (scientific notation) ile yazdır
print(f"L2 Error (Unified):       {err_unified:.6e}")
print(f"L2 Error (Decomposition): {err_decomp:.6e}")


L2 Error (Unified):       1.987954e-02
L2 Error (Decomposition): 1.758157e-16
